In [ ]:
from pathlib import Path
WORK_ROOT = Path('/content')
print('WORK_ROOT:', WORK_ROOT)

In [ ]:
import torch
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU が有効になっていません。Runtime > Change runtime type > GPU を確認してください。')

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
    print('Token loaded from Colab Secrets.')
except Exception:
    os.environ['GITHUB_TOKEN'] = 'ここに入力する'
    pass

TOKEN = os.environ.get('GITHUB_TOKEN', '')
assert TOKEN, 'GITHUB_TOKEN が設定されていません。上のセルを確認してください。'
print('GITHUB_TOKEN: OK')

In [ ]:
import os, base64, subprocess
REPO_DIR = WORK_ROOT / 'Damaten'
TOKEN = os.environ['GITHUB_TOKEN']

# Inject an Authorization header for github.com that BOTH git and git-lfs use
# (the GitHub Actions approach). No credential lookup, and the token never
# appears in a URL or in the logs. Set via subprocess so it is not echoed.
basic = base64.b64encode(f'x-access-token:{TOKEN}'.encode()).decode()
subprocess.run(['git', 'config', '--global', 'http.https://github.com/.extraheader',
                f'AUTHORIZATION: basic {basic}'], check=True)

CLONE_URL = 'https://github.com/Koushien552/Damaten.git'   # no token in the URL
if not REPO_DIR.exists():
    !git clone "{CLONE_URL}" "{REPO_DIR}"
%cd "{REPO_DIR}"
!git remote set-url origin "{CLONE_URL}"
!git fetch origin main
!git reset --hard FETCH_HEAD
!git lfs pull
print('Clone/pull: OK')

In [ ]:
!python "{REPO_DIR}/colab/github_gpu_train_and_push.py" \
  --work-root "{WORK_ROOT}" \
  --repo-url https://github.com/Koushien552/Damaten.git \
  --branch main \
  --n 9 \
  --arch cnn \
  --channels 32 --blocks 4 --symmetry full